# Lagrange points of the restricted three-body problem

*astro · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/lagrange_points)

A star and a planet circle each other; in their rotating frame a test particle feels gravity, the centrifugal force and the Coriolis force, written with $\varepsilon_{ijk}$. Newton's method on $\nabla\Phi_\mathrm{eff} = 0$ (gradient and Hessian by autograd) finds all five Lagrange points at once, and particles nudged away from L4 trace tadpole orbits around it.

**Vector form**

$$\ddot{\mathbf r} = -\nabla\Phi_\mathrm{grav} - 2\,\boldsymbol\Omega\times\dot{\mathbf r} - \boldsymbol\Omega\times(\boldsymbol\Omega\times\mathbf r),\qquad \nabla\Phi_\mathrm{eff}(\mathbf r_L) = 0$$

**Index (tensor) form**

$$\ddot x_i = -m_s\,\frac{(x - x_s)_i}{\lvert x - x_s\rvert^3} - 2\,\varepsilon_{ijk}\Omega_j\dot x_k - \varepsilon_{ijk}\Omega_j\,\varepsilon_{klm}\Omega_l x_m,\qquad x_i \leftarrow x_i - (\partial_i\partial_j\Phi_\mathrm{eff})^{-1}\partial_j\Phi_\mathrm{eff}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bi}$ | `y[:, 0, :]` | (B, 3) | position of test particle $b$ in the rotating frame |
| $x_{si},\; m_s$ | `primaries(mu)` | (2, 3), (2,) | star at $(-\mu, 0, 0)$ and planet at $(1-\mu, 0, 0)$ |
| $m_s (x - x_s)_i / r^3$ | `einsum('s,bs,bsi->bi', m, 1/r3, d)` | (B, 3) | repeated $s$ ⇒ summed: gravity of both bodies |
| $\varepsilon_{ijk}$ | `levi_civita()` | (3, 3, 3) | Levi-Civita symbol for the cross products |
| $-2\varepsilon_{ijk}\Omega_j\dot x_k$ | `-2 * einsum('ijk,j,bk->bi', eps, omega, v)` | (B, 3) | Coriolis acceleration, $\boldsymbol\Omega = \hat{\mathbf z}$ |
| $\partial_i\Phi_\mathrm{eff}$ | `vmap(grad(effective_potential))(x)` | (5, 3) | gradient by autograd at all five points |
| $\partial_i\partial_j\Phi_\mathrm{eff}$ | `vmap(jacrev(grad(effective_potential)))(x)` | (5, 3, 3) | Hessian by autograd: Newton's step is one batched linalg.solve |
| $C$ | `jacobi_constant(y, mu)` | (T, B) | $-2\Phi_\mathrm{eff} - \dot x_i\dot x_i$, conserved |

Units: separation 1, $G(M_1 + M_2) = 1$, so the pair turns once every $2\pi$. The planet has mass fraction $\mu$, and the test particles are too light to disturb either body. L1, L2 and L3 lie on the line through the bodies and are saddle points of $\Phi_\mathrm{eff}$; L4 and L5 form equilateral triangles with the two bodies and are maxima of $\Phi_\mathrm{eff}$, yet the Coriolis force makes them stable when $\mu < 0.0385$. That is why Jupiter's Trojan asteroids gather 60° ahead of and behind the planet. The particles start at rest on the planet's orbit, a few degrees ahead of L4: small nudges give small tadpole loops (with fast epicycles on top), larger ones stretch into long banana-shaped loops along the orbit, and beyond about 20° (for $\mu = 0.01$) the particle wanders off. For $\mu$ above 0.0385 even tiny nudges drift away.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/lagrange_points/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    mu = params["mu"]
    points = solve.lagrange_points(tensors.lagrange_guesses(mu, dev), mu)
    residual = equation.grad_potential(points, mu).norm(dim=-1)  # (5,) |∇Φ_eff| after Newton
    phi_points = equation.effective_potential(points, mu)

    nudges = [params["nudge"] * f for f in (0.25, 0.6, 1.0)]  # degrees ahead of L4
    y0 = tensors.particles_near_l4(mu, nudges, dev)
    times, traj = solve.solve(y0, mu, params["t_max"], params["dt"])
    C = equation.jacobi_constant(traj, mu)  # (T, B)

    for k in range(5):
        x, y = points[k, 0].item(), points[k, 1].item()
        print(f"L{k + 1}: ({x:+.5f}, {y:+.5f})   |∇Φ_eff| = {residual[k]:.1e}   Φ_eff = {phi_points[k]:.5f}")
    star, planet = points.new_tensor([-mu, 0, 0]), points.new_tensor([1 - mu, 0, 0])
    d_star, d_planet = (points[3] - star).norm(), (points[3] - planet).norm()
    print(f"L4 distances to star and planet: {d_star:.6f}, {d_planet:.6f} (equilateral triangle: both 1)")
    print(f"Jacobi constant drift over the run: {(C - C[0]).abs().max():.1e}")

    grid = tensors.potential_grid(240, 1.6, dev)
    phi = equation.effective_potential(grid, mu)
    return plot.animate(times, traj, points, grid, phi, phi_points, mu, nudges, out_dir / "lagrange_points.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/lagrange_points/equation.py
r"""Physics → tensor translation.

Circular restricted three-body problem: a star (mass 1 − μ) and a planet (mass μ) circle their centre of mass,
and a test particle feels both. In the frame rotating with the pair (angular velocity Ω = ẑ) the two masses sit
still at x_1 = (−μ, 0, 0) and x_2 = (1 − μ, 0, 0). Units: G(M_1 + M_2) = 1, separation 1, Ω = 1.

The effective potential adds the centrifugal term to gravity:

    Φ_eff = -\frac{1-μ}{|x - x_1|} - \frac{μ}{|x - x_2|} - ½\left(x_j x_j - (Ω_j x_j)^2\right)

and the equation of motion in the rotating frame has gravity, Coriolis and centrifugal forces,
written with the Levi-Civita symbol ε_{ijk} (all repeated indices summed):

    \ddot x_i = -\sum_s m_s \frac{(x - x_s)_i}{|x - x_s|^3}
                - 2\,ε_{ijk} Ω_j \dot x_k
                - ε_{ijk} Ω_j\, ε_{klm} Ω_l x_m

Lagrange points are where a particle can sit at rest: ∂_i Φ_eff = 0. Newton's method finds them,

    x ← x - (∂_i ∂_j Φ_eff)^{-1}\, ∂_j Φ_eff,

with the gradient and Hessian from torch.func (automatic differentiation). The Jacobi constant
C = -2Φ_eff - \dot x_i \dot x_i is conserved along every orbit.
"""

import torch
from torch.func import grad, jacrev, vmap


def levi_civita(dtype=torch.float64, device=None) -> torch.Tensor:
    """ε_ijk, shape (3, 3, 3)."""
    eps = torch.zeros(3, 3, 3, dtype=dtype, device=device)
    for i, j, k in [(0, 1, 2), (1, 2, 0), (2, 0, 1)]:
        eps[i, j, k], eps[i, k, j] = 1.0, -1.0
    return eps


def primaries(mu: float, dtype=torch.float64, device=None) -> tuple[torch.Tensor, torch.Tensor]:
    """Positions x_s (2, 3) and masses m_s (2,) of the star and the planet in the rotating frame."""
    pos = torch.tensor([[-mu, 0.0, 0.0], [1 - mu, 0.0, 0.0]], dtype=dtype, device=device)
    return pos, torch.tensor([1 - mu, mu], dtype=dtype, device=device)


def effective_potential(x: torch.Tensor, mu: float) -> torch.Tensor:
    r"""Φ_eff at points x (..., 3) → (...). Ω = ẑ, so (Ω_j x_j)² = z²."""
    pos, m = primaries(mu, x.dtype, x.device)
    d = x[..., None, :] - pos  # (..., 2, 3) separation from each mass s
    r = torch.sqrt(torch.einsum("...sj,...sj->...s", d, d))  # (..., 2)
    gravity = -torch.einsum("s,...s->...", m, 1 / r)
    centrifugal = -0.5 * (torch.einsum("...j,...j->...", x, x) - x[..., 2] ** 2)
    return gravity + centrifugal


def grad_potential(x: torch.Tensor, mu: float) -> torch.Tensor:
    """∂_i Φ_eff by automatic differentiation, for a batch of points.   (P, 3) → (P, 3)"""
    return vmap(grad(lambda p: effective_potential(p, mu)))(x)


def hessian_potential(x: torch.Tensor, mu: float) -> torch.Tensor:
    """∂_i ∂_j Φ_eff by automatic differentiation.   (P, 3) → (P, 3, 3)"""
    return vmap(jacrev(grad(lambda p: effective_potential(p, mu))))(x)  # reverse mode twice


def acceleration(x: torch.Tensor, v: torch.Tensor, mu: float, eps: torch.Tensor) -> torch.Tensor:
    r"""\ddot x_i = gravity - 2 ε_{ijk} Ω_j v_k - ε_{ijk} Ω_j ε_{klm} Ω_l x_m.   x, v: (B, 3) → (B, 3)"""
    pos, m = primaries(mu, x.dtype, x.device)
    omega = torch.tensor([0.0, 0.0, 1.0], dtype=x.dtype, device=x.device)
    d = x[:, None, :] - pos  # (B, 2, 3)
    r3 = torch.einsum("bsj,bsj->bs", d, d) ** 1.5  # (B, 2)
    gravity = -torch.einsum("s,bs,bsi->bi", m, 1 / r3, d)
    coriolis = -2 * torch.einsum("ijk,j,bk->bi", eps, omega, v)
    omega_cross_x = torch.einsum("klm,l,bm->bk", eps, omega, x)
    centrifugal = -torch.einsum("ijk,j,bk->bi", eps, omega, omega_cross_x)
    return gravity + coriolis + centrifugal


def rhs(y: torch.Tensor, mu: float, eps: torch.Tensor) -> torch.Tensor:
    """dy/dt for the state y (B, 2, 3): y[:, 0] = position, y[:, 1] = velocity (rotating frame)."""
    return torch.stack([y[:, 1], acceleration(y[:, 0], y[:, 1], mu, eps)], dim=1)


def jacobi_constant(y: torch.Tensor, mu: float) -> torch.Tensor:
    r"""C = -2Φ_eff - \dot x_i \dot x_i.   y: (..., 2, 3) → (...)"""
    x, v = y[..., 0, :], y[..., 1, :]
    return -2 * effective_potential(x, mu) - torch.einsum("...i,...i->...", v, v)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/lagrange_points/tensors.py
"""Build the concrete tensors: first guesses for the five Lagrange points and the test particles near L4."""

import math

import torch


def lagrange_guesses(mu: float, device: torch.device) -> torch.Tensor:
    """Rough positions (5, 3) of L1 … L5 (Hill-sphere estimates for L1/L2, triangles for L4/L5)."""
    h = (mu / 3) ** (1 / 3)
    s = math.sqrt(3) / 2
    return torch.tensor(
        [
            [1 - mu - h, 0.0, 0.0],  # L1 between the bodies
            [1 - mu + h, 0.0, 0.0],  # L2 beyond the planet
            [-1 - 5 * mu / 12, 0.0, 0.0],  # L3 opposite the planet
            [0.5 - mu, s, 0.0],  # L4 leading by 60°
            [0.5 - mu, -s, 0.0],  # L5 trailing by 60°
        ],
        dtype=torch.float64,
        device=device,
    )


def particles_near_l4(mu: float, nudges_deg: list[float], device: torch.device) -> torch.Tensor:
    """Initial states (B, 2, 3): particles at rest in the rotating frame on the unit circle around the star,
    nudged ahead of L4 (which sits 60° ahead of the planet) by the given angles."""
    theta = torch.deg2rad(60.0 + torch.tensor(nudges_deg, dtype=torch.float64, device=device))  # (B,)
    x = torch.stack([-mu + torch.cos(theta), torch.sin(theta), torch.zeros_like(theta)], dim=-1)  # (B, 3)
    return torch.stack([x, torch.zeros_like(x)], dim=1)


def potential_grid(n: int, reach: float, device: torch.device) -> torch.Tensor:
    """Points (n, n, 3) of the z = 0 plane for the contour plot."""
    s = torch.linspace(-reach, reach, n, dtype=torch.float64, device=device)
    X, Y = torch.meshgrid(s, s, indexing="xy")
    return torch.stack([X, Y, torch.zeros_like(X)], dim=-1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/lagrange_points/solve.py
"""Find the Lagrange points with Newton's method, then integrate test particles near L4 with RK4."""

import torch

from problems._common.integrators import integrate

from .equation import grad_potential, hessian_potential, levi_civita, rhs


def lagrange_points(guess: torch.Tensor, mu: float, iterations: int = 30) -> torch.Tensor:
    """Newton iterations x ← x − H⁻¹∇Φ_eff for all five points at once: (5, 3) → (5, 3)."""
    x = guess.clone()
    for _ in range(iterations):
        g = grad_potential(x, mu)  # (5, 3)
        H = hessian_potential(x, mu)  # (5, 3, 3)
        x = x - torch.linalg.solve(H, g)  # one batched linear solve per point
    return x


def solve(y0: torch.Tensor, mu: float, t_max: float, dt: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), trajectory (T, B, 2, 3)) in the rotating frame."""
    eps = levi_civita(y0.dtype, y0.device)
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda y, t: rhs(y, mu, eps), y0, dt, n_steps)
    times = torch.arange(n_steps + 1, dtype=y0.dtype, device=y0.device) * dt
    return times, traj


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/lagrange_points/plot.py
"""Contours of the effective potential with the five Lagrange points, and particles librating around L4."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    times: torch.Tensor,
    traj: torch.Tensor,
    points: torch.Tensor,
    grid: torch.Tensor,
    phi: torch.Tensor,
    phi_points: torch.Tensor,
    mu: float,
    nudges: list[float],
    path: Path,
) -> Path:
    t = times.cpu().numpy()
    pos = traj[:, :, 0, :2].cpu().numpy()  # (T, B, 2)
    L = points[:, :2].cpu().numpy()  # (5, 2)
    X, Y = grid[..., 0].cpu().numpy(), grid[..., 1].cpu().numpy()
    phi = phi.cpu().numpy()
    phi_L = np.unique(phi_points.cpu().numpy())  # contour levels through the five L points
    levels = np.unique(np.concatenate([np.linspace(phi_L[0] - 0.4, phi_L[0], 5), phi_L, [phi_L[-1] + 0.01]]))

    fig, (ax, ax_z) = plt.subplots(1, 2, figsize=(8, 4))
    for a in (ax, ax_z):
        a.contourf(X, Y, np.clip(phi, levels[0], levels[-1]), levels=levels, cmap="Blues_r", alpha=0.4)
        a.contour(X, Y, phi, levels=phi_L, colors="#6b7280", linewidths=0.6)
        a.set_aspect("equal", adjustable="box")
        a.plot([-mu], [0], "o", ms=12, color="#f59e0b", zorder=5)  # star
        a.plot([1 - mu], [0], "o", ms=5, color="#1e40af", zorder=5)  # planet
        a.plot(L[:, 0], L[:, 1], "x", ms=7, mew=2, color=ACCENT[1], zorder=5)
        a.grid(False)
    for k, (x, y) in enumerate(L):
        ax.annotate(f"L{k + 1}", (x, y), textcoords="offset points", xytext=(4, 5), fontsize=8, color=ACCENT[1])
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel("x (rotating frame)")
    ax.set_ylabel("y")
    ax.set_title(f"Φ_eff contours, μ = {mu:g}", fontsize=9)
    zoom = 0.45
    cx, cy = L[3]
    ax_z.set_xlim(cx - zoom, cx + zoom)
    ax_z.set_ylim(cy - zoom, cy + zoom)
    ax_z.set_title("zoom on L4: tadpole orbits", fontsize=9)

    colors = [ACCENT[(b + 2) % len(ACCENT)] for b in range(pos.shape[1])]
    trails = [
        [a.plot([], [], lw=1.0, color=c, label=f"nudged {o:.1f}°")[0] for a in (ax, ax_z)]
        for c, o in zip(colors, nudges)
    ]
    dots = [[a.plot([], [], "o", ms=4, color=c)[0] for a in (ax, ax_z)] for c in colors]
    ax_z.legend(fontsize=7, loc="lower left")
    clock = ax.text(0.03, 0.96, "", transform=ax.transAxes, va="top", fontsize=9)

    def update(step: int):
        for b in range(pos.shape[1]):
            for line in trails[b]:
                line.set_data(pos[: step + 1, b, 0], pos[: step + 1, b, 1])
            for dot in dots[b]:
                dot.set_data([pos[step, b, 0]], [pos[step, b, 1]])
        clock.set_text(f"t = {t[step]:.0f}   ({t[step] / (2 * np.pi):.1f} planet orbits)")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'mu': 0.01,
    'nudge': 15.0,
    't_max': 100.0,
    'dt': 0.03
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.lagrange_points")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.lagrange_points.main")
problem_dir = Path("problems/astro/lagrange_points")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
